In [ ]:
import numpy as np
import xarray as xr
from datetime import datetime, timedelta
# from scipy import stats
import pickle
import json

In [ ]:
# for plotting

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc

sns.set()
sns.set_context('poster')
sns.set_style('ticks')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'cmr10'
plt.rcParams["mathtext.fontset"] = 'cm'
plt.rcParams["axes.formatter.use_mathtext"] = True

In [ ]:
nx, ny, nz, nt = 512, 512, 120, 241
dts = 0.5 # minute
dx = 25 # m
dy = 25 # m
dz = 25 # m
grid_vol = dx*1.0e-3*dy*1.0e-3*dz*1.0e-3 # km**3
z = np.arange(dz/2, 3000., dz)
t = np.arange(0, 241)*dts # minutes
ti = np.arange(-0.5, 241., 1.)*dts
zi = np.arange(0., 3001., dz)

In [ ]:
ctl = 'bomex_25m_r20251009'
ehe18 = 'bomex_25m_ehe18_r20251009'

In [ ]:
def total_wq(casename):
    # with open(f'{casename}/pkl/qclm.pkl', 'rb') as f:
    #     qclm = pickle.load(f)
    with open(f'{casename}/pkl/ws.pkl', 'rb') as f:
        ws = pickle.load(f)
    with open(f'{casename}/pkl/qts.pkl', 'rb') as f:
        qts = pickle.load(f)
    qfluxes = []
    qd = [
        ["MUa", "MDa", "DDa", "DUa", "ENa"],
        ["MUatl", "MDatl", "DDatl", "DUatl", "ENatl"],
        ["MUatul", "MDatul", "DDatul", "DUatul", "ENatul"],
        ["MUatula", "MDatula", "DDatula", "DUatula", 'ENatula'],
    ]
    for quads in qd:
        qflux = []
        for quad in quads:
            with open(f'{casename}/pkl/{quad}_qflux.pkl', 'rb') as f:
                qflux.append(pickle.load(f)*ws*qts)
        qfluxes.append(qflux)
    return np.asarray(qfluxes)

In [ ]:
qfluxes = {}
qfluxes[ctl] = total_wq(ctl)
qfluxes[ehe18] = total_wq(ehe18)

In [ ]:
total_wq = {}
total_wq[ctl] = qfluxes[ctl].sum(axis=1)
total_wq[ehe18] = qfluxes[ehe18].sum(axis=1)

In [ ]:
pop_labels = ['domain', 'tracked', 'tracked full', 'tracked full attached']
styles = ['--', ':', '-', 'None']
markers = [None, None, None, '+']

In [ ]:
fig = plt.figure(figsize=(6, 9))
ax = fig.add_axes((0.15, 0.1, 0.8, 0.85))
for i in range(4):
    ax.plot(total_wq[ctl][i]*1.0e3, z*1.0e-3, linestyle=styles[i], marker=markers[i], label=f'CTL - {pop_labels[i]}', color='black')
    ax.plot(total_wq[ehe18][i]*1.0e3, z*1.0e-3, linestyle=styles[i], marker=markers[i], label=f'EHE18 - {pop_labels[i]}', color='red')
ax.set_xlabel('Time (min)')
ax.set_ylim((0, 2.5))
ax.set_ylabel('Height (km)')
# ax.set_xlim((-10, 2))
ax.set_xlabel(r'($10^{-3}$ g/kg m/s)')
plt.legend(loc='upper right', fontsize=14)
plt.show()

In [ ]:
total_wq_diff = total_wq[ehe18] - total_wq[ctl]
fig = plt.figure(figsize=(6, 9))
ax = fig.add_axes((0.15, 0.1, 0.8, 0.85))
for i in range(4):
    ax.plot(total_wq_diff[i]*1.0e3, z*1.0e-3, linestyle=styles[i], marker=markers[i], label=f'Diff - {pop_labels[i]}', color='black')
ax.set_xlabel('Time (min)')
ax.set_ylim((0, 2.5))
ax.set_ylabel('Height (km)')
# ax.set_xlim((-10, 2))
ax.set_xlabel(r'($10^{-3}$ g/kg m/s)')
plt.legend(loc='upper left', fontsize=14)
plt.show()